# 3D 전처리 & U-Net 학습

In [ ]:
import os
import shutil
import glob
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm import tqdm

# 1. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 초기 데이터 압축 해제
hipo_folder = '/content/drive/MyDrive/hipo'
tar_path = os.path.join(hipo_folder, 'Task04_Hippocampus.tar')
extract_path_initial = os.path.join(hipo_folder, 'Task04_Hippocampus_extracted')
shutil.unpack_archive(tar_path, extract_path_initial)
print("📦 Task04_Hippocampus.tar 압축 해제 완료!")

In [ ]:
# 3. 상세 데이터 경로 설정
extract_path = os.path.join(extract_path_initial, 'Task04_Hippocampus')
imagesTr_raw_dir = os.path.join(extract_path, 'imagesTr')
labelsTr_raw_dir = os.path.join(extract_path, 'labelsTr')
imagesTs_raw_dir = os.path.join(extract_path, 'imagesTs')

img_tr_paths = sorted(glob.glob(os.path.join(imagesTr_raw_dir, '*.nii*')))
lbl_tr_paths = sorted(glob.glob(os.path.join(labelsTr_raw_dir, '*.nii*')))
img_ts_paths = sorted(glob.glob(os.path.join(imagesTs_raw_dir, '*.nii*')))

print(f"🧠 imagesTr 개수: {len(img_tr_paths)}")
print(f"🏷 labelsTr 개수: {len(lbl_tr_paths)}")
print(f"🧪 imagesTs 개수: {len(img_ts_paths)}")

In [ ]:
# 4. 유틸리티 함수 정의
def percentile_clipping(image, lower=0.5, upper=99.5):
    lo = np.percentile(image, lower)
    hi = np.percentile(image, upper)
    return np.clip(image, lo, hi)

def zscore_normalize(image, eps=1e-8):
    mask = image > 0
    if mask.sum() == 0:
        return image
    mean = image[mask].mean()
    std = image[mask].std()
    return (image - mean) / (std + eps)

def crop_to_roi(image, label, margin=10):
    coords = np.where(label > 0)
    if coords[0].size == 0:
        return image, label

    zmin, ymin, xmin = coords[0].min(), coords[1].min(), coords[2].min()
    zmax, ymax, xmax = coords[0].max(), coords[1].max(), coords[2].max()

    zmin = max(0, zmin - margin)
    ymin = max(0, ymin - margin)
    xmin = max(0, xmin - margin)
    zmax = min(image.shape[0], zmax + margin)
    ymax = min(image.shape[1], ymax + margin)
    xmax = min(image.shape[2], xmax + margin)

    roi_image = image[zmin:zmax, ymin:ymax, xmin:xmax]
    roi_label = label[zmin:zmax, ymin:ymax, xmin:xmax]
    roi_label = np.rint(roi_label).astype(np.int64)

    return roi_image, roi_label

In [ ]:
# 5. 데이터 전처리 파이프라인
images_clipped_dir = os.path.join(extract_path, 'imagesTr_clipped')
images_norm_dir = os.path.join(extract_path, 'imagesTr_clipped_norm')
images_roi_dir = os.path.join(extract_path, 'imagesTr_roi')
labels_roi_dir = os.path.join(extract_path, 'labelsTr_roi')

os.makedirs(images_clipped_dir, exist_ok=True)
os.makedirs(images_norm_dir, exist_ok=True)
os.makedirs(images_roi_dir, exist_ok=True)
os.makedirs(labels_roi_dir, exist_ok=True)

print("--- Percentile Clipping 적용 중 ---")
for img_path_raw in tqdm(img_tr_paths, desc="Clipping Images"):
    fname = os.path.basename(img_path_raw)
    nii = nib.load(img_path_raw)
    image_data = nii.get_fdata()
    clipped_image_data = percentile_clipping(image_data)
    nib.save(nib.Nifti1Image(clipped_image_data, nii.affine, nii.header),
             os.path.join(images_clipped_dir, fname))

print("--- Z-score Normalization 적용 중 ---")
clipped_img_paths = sorted(glob.glob(os.path.join(images_clipped_dir, '*.nii*')))
for img_path_clipped in tqdm(clipped_img_paths, desc="Normalizing Images"):
    fname = os.path.basename(img_path_clipped)
    nii = nib.load(img_path_clipped)
    image_data = nii.get_fdata()
    norm_image_data = zscore_normalize(image_data)
    nib.save(nib.Nifti1Image(norm_image_data, nii.affine, nii.header),
             os.path.join(images_norm_dir, fname))

print("--- ROI Cropping 및 저장 중 ---")
norm_img_paths = sorted(glob.glob(os.path.join(images_norm_dir, '*.nii*')))
for img_path_norm in tqdm(norm_img_paths, desc="Cropping ROIs"):
    fname = os.path.basename(img_path_norm)
    lbl_path_raw = os.path.join(labelsTr_raw_dir, fname)
    if not os.path.exists(lbl_path_raw):
        print(f"Warning: {fname}에 대한 라벨 파일을 찾을 수 없습니다.")
        continue

    img_nii = nib.load(img_path_norm)
    lbl_nii = nib.load(lbl_path_raw)
    image_data = img_nii.get_fdata()
    label_data = lbl_nii.get_fdata()
    roi_image_data, roi_label_data = crop_to_roi(image_data, label_data)

    nib.save(nib.Nifti1Image(roi_image_data, img_nii.affine, img_nii.header),
             os.path.join(images_roi_dir, fname))
    nib.save(nib.Nifti1Image(roi_label_data, lbl_nii.affine, lbl_nii.header),
             os.path.join(labels_roi_dir, fname))
print("✅ 데이터 전처리 완료!")

In [ ]:
# 6. 개선된 Dataset 구현 (더 큰 타겟 사이즈 + 패딩)
class SliceDataset(Dataset):
    def __init__(self, image_dir, label_dir, target_size=(256, 256)):
        self.image_files = sorted([f for f in os.listdir(image_dir)
                                   if f.endswith('.nii.gz') and not f.startswith('._')])
        self.label_files = sorted([f for f in os.listdir(label_dir)
                                   if f.endswith('.nii.gz') and not f.startswith('._')])
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.target_size = target_size

        if len(self.image_files) != len(self.label_files):
            raise ValueError("이미지와 라벨 파일 개수 불일치")

        self.data_pairs = []
        all_labels = []

        for i in range(len(self.image_files)):
            img_path = os.path.join(self.image_dir, self.image_files[i])
            lbl_path = os.path.join(self.label_dir, self.label_files[i])
            img_nii = nib.load(img_path)
            lbl_nii = nib.load(lbl_path)
            image_3d = img_nii.get_fdata()
            label_3d = lbl_nii.get_fdata()

            # 2D 슬라이스 추출 (깊이 방향)
            for z in range(image_3d.shape[2]):
                slice_img = image_3d[:, :, z]
                slice_lbl = label_3d[:, :, z]

                # 해마가 존재하는 슬라이스만 포함
                if np.sum(slice_lbl > 0) > 0:
                    self.data_pairs.append((slice_img, slice_lbl))
                    all_labels.extend(slice_lbl.flatten())

        # 클래스 분포 확인
        label_counts = Counter(all_labels)
        print(f"\n✅ 데이터셋 생성 완료: 총 {len(self.data_pairs)} 슬라이스")
        print(f"   라벨 분포: {dict(label_counts)}")

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        image, label = self.data_pairs[idx]

        # Tensor 변환
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)  # (1, H, W)
        label = torch.tensor(label, dtype=torch.long)  # (H, W)

        # Resize with proper interpolation
        image = image.unsqueeze(0)  # (1, 1, H, W)
        label = label.unsqueeze(0).unsqueeze(0).float()  # (1, 1, H, W)

        # 더 큰 크기로 resize하여 작은 클래스 보존
        image = F.interpolate(image, size=self.target_size, mode='bilinear', align_corners=False)
        label = F.interpolate(label, size=self.target_size, mode='nearest')

        image = image.squeeze(0)  # (1, H, W)
        label = label.squeeze(0).squeeze(0).long()  # (H, W)

        return image, label

In [ ]:
# 개선된 collate_fn (이미 같은 크기이므로 단순화)
def custom_collate_fn(batch):
    images = []
    labels = []

    for img, lbl in batch:
        images.append(img)
        labels.append(lbl)

    images = torch.stack(images)  # (B, 1, H, W)
    labels = torch.stack(labels)  # (B, H, W)

    return images, labels

In [ ]:
# 7. 데이터셋 및 DataLoader 생성
print("\n--- Dataset 및 DataLoader 생성 중 ---")
dataset = SliceDataset(image_dir=images_roi_dir, label_dir=labels_roi_dir, target_size=(256, 256))
train_loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)

In [ ]:
# 배치 확인 및 라벨 분포 체크
for images, labels in train_loader:
    print(f"\n📊 배치 정보:")
    print(f"   이미지 shape: {images.shape}")
    print(f"   라벨 shape: {labels.shape}")
    unique_labels = torch.unique(labels)
    print(f"   배치 내 고유 라벨 값: {unique_labels.tolist()}")
    break

In [ ]:
# 8. U-Net 모델 정의
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=3):
        super(UNet, self).__init__()

        def CBR(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.enc1_1 = CBR(in_channels, 64)
        self.enc1_2 = CBR(64, 64)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.enc2_1 = CBR(64, 128)
        self.enc2_2 = CBR(128, 128)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.enc3_1 = CBR(128, 256)
        self.enc3_2 = CBR(256, 256)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.enc4_1 = CBR(256, 512)
        self.enc4_2 = CBR(512, 512)
        self.pool4 = nn.MaxPool2d(2, 2)

        # Center
        self.center1 = CBR(512, 1024)
        self.center2 = CBR(1024, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4_1 = CBR(1024, 512)
        self.dec4_2 = CBR(512, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3_1 = CBR(512, 256)
        self.dec3_2 = CBR(256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2_1 = CBR(256, 128)
        self.dec2_2 = CBR(128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1_1 = CBR(128, 64)
        self.dec1_2 = CBR(64, 64)

        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        # Encoder
        enc1 = self.enc1_2(self.enc1_1(x))
        pool1 = self.pool1(enc1)

        enc2 = self.enc2_2(self.enc2_1(pool1))
        pool2 = self.pool2(enc2)

        enc3 = self.enc3_2(self.enc3_1(pool2))
        pool3 = self.pool3(enc3)

        enc4 = self.enc4_2(self.enc4_1(pool3))
        pool4 = self.pool4(enc4)

        # Center
        center = self.center2(self.center1(pool4))

        # Decoder
        up4 = self.up4(center)
        dec4 = self.dec4_2(self.dec4_1(torch.cat([up4, enc4], dim=1)))

        up3 = self.up3(dec4)
        dec3 = self.dec3_2(self.dec3_1(torch.cat([up3, enc3], dim=1)))

        up2 = self.up2(dec3)
        dec2 = self.dec2_2(self.dec2_1(torch.cat([up2, enc2], dim=1)))

        up1 = self.up1(dec2)
        dec1 = self.dec1_2(self.dec1_1(torch.cat([up1, enc1], dim=1)))

        return self.final(dec1)

In [ ]:
# 9. 모델 초기화 및 설정
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n🖥️ 사용 디바이스: {device}")

model = UNet(in_channels=1, out_channels=3).to(device)
print(f"✅ U-Net 모델 생성 완료")
print(f"   총 파라미터: {sum(p.numel() for p in model.parameters()):,}개")

In [ ]:
# 클래스 가중치 계산 (클래스 불균형 처리)
all_labels_for_weight = []
for _, lbl in dataset:
    all_labels_for_weight.extend(lbl.numpy().flatten())

label_counts = Counter(all_labels_for_weight)
total_pixels = sum(label_counts.values())
class_weights = []
for i in range(3):
    if i in label_counts:
        weight = total_pixels / (3 * label_counts[i])
    else:
        weight = 1.0
    class_weights.append(weight)

class_weights = torch.FloatTensor(class_weights).to(device)
print(f"\n⚖️ 클래스 가중치: {class_weights.tolist()}")

In [ ]:
# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# Dice Score 함수
def dice_score(pred, target, num_classes=3, epsilon=1e-6):
    pred_classes = torch.argmax(pred, dim=1)  # (B, H, W)
    dices = []

    for c in range(num_classes):
        pred_c = (pred_classes == c).float()
        target_c = (target == c).float()
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        dice = (2. * intersection + epsilon) / (union + epsilon)
        dices.append(dice.item())

    return dices

In [ ]:
# 10. 학습 루프
num_epochs = 20

print("\n" + "="*60)
print("🚀 학습 시작")
print("="*60)

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    epoch_dice = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        batch_dice = dice_score(outputs, labels)
        epoch_dice.append(batch_dice)

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = epoch_loss / len(train_loader)
    avg_dice = np.mean(epoch_dice, axis=0)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"  Loss: {avg_loss:.4f}")
    print(f"  Dice [BG, L_Hippo, R_Hippo]: [{avg_dice[0]:.4f}, {avg_dice[1]:.4f}, {avg_dice[2]:.4f}]")
    print(f"  Mean Dice (Hippo): {np.mean(avg_dice[1:]):.4f}")

print("\n" + "="*60)
print("✅ 학습 완료")
print("="*60)

# 11. 최종 평가
print("\n--- 최종 평가 중 ---")
model.eval()
all_dice = []

with torch.no_grad():
    for images, labels in tqdm(train_loader, desc="Evaluation"):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        dices = dice_score(outputs, labels)
        all_dice.append(dices)

mean_dice = np.mean(all_dice, axis=0)
print(f"\n📊 최종 Dice Score:")
print(f"   Background: {mean_dice[0]:.4f}")
print(f"   Left Hippocampus: {mean_dice[1]:.4f}")
print(f"   Right Hippocampus: {mean_dice[2]:.4f}")
print(f"   Mean (Hippocampi): {np.mean(mean_dice[1:]):.4f}")

# 2D U-Net 수행
## 2D U-Net metrix 시각화 & Dice 시각화 & 이미지 시각화 코드

In [ ]:
import os
import shutil
import glob
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [ ]:
# 1. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 데이터 압축 해제
hipo_folder = '/content/drive/MyDrive/hipo'
tar_path = os.path.join(hipo_folder, 'Task04_Hippocampus.tar')
extract_path_initial = os.path.join(hipo_folder, 'Task04_Hippocampus_extracted')

if not os.path.exists(extract_path_initial):
    shutil.unpack_archive(tar_path, extract_path_initial)
    print("📦 Task04_Hippocampus.tar 압축 해제 완료!")
else:
    print("📦 이미 압축 해제된 데이터 사용")

In [ ]:
# 3. 데이터 경로 설정
extract_path = os.path.join(extract_path_initial, 'Task04_Hippocampus')
imagesTr_dir = os.path.join(extract_path, 'imagesTr')
labelsTr_dir = os.path.join(extract_path, 'labelsTr')
imagesTs_dir = os.path.join(extract_path, 'imagesTs')

img_tr_paths = sorted(glob.glob(os.path.join(imagesTr_dir, '*.nii*')))
lbl_tr_paths = sorted(glob.glob(os.path.join(labelsTr_dir, '*.nii*')))

print(f"🧠 imagesTr 개수: {len(img_tr_paths)}")
print(f"🏷 labelsTr 개수: {len(lbl_tr_paths)}")

In [ ]:
# 4. 순수 2D U-Net용 Dataset (전처리 없음)
class Pure2DSliceDataset(Dataset):
    def __init__(self, image_paths, label_paths, target_size=(256, 256)):
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.target_size = target_size

        if len(image_paths) != len(label_paths):
            raise ValueError("이미지와 라벨 파일 개수 불일치")

        self.data_pairs = []
        all_labels = []

        print("\n🔍 3D 볼륨에서 2D 슬라이스 추출 중...")
        for img_path, lbl_path in tqdm(zip(image_paths, label_paths), total=len(image_paths)):
            # NIfTI 파일 로드
            img_nii = nib.load(img_path)
            lbl_nii = nib.load(lbl_path)

            img_volume = img_nii.get_fdata()  # (H, W, D)
            lbl_volume = lbl_nii.get_fdata()  # (H, W, D)

            # 각 슬라이스 추출 (depth 방향)
            for z in range(img_volume.shape[2]):
                img_slice = img_volume[:, :, z]
                lbl_slice = lbl_volume[:, :, z]

                # 해마가 존재하는 슬라이스만 포함
                if np.sum(lbl_slice > 0) > 0:
                    # 간단한 정규화만 적용 (zero mean, unit variance)
                    if img_slice.max() > 0:
                        img_slice = (img_slice - img_slice.mean()) / (img_slice.std() + 1e-8)

                    self.data_pairs.append((img_slice, lbl_slice))
                    all_labels.extend(lbl_slice.flatten())

        # 클래스 분포 확인
        label_counts = Counter(all_labels)
        print(f"\n✅ 데이터셋 생성 완료: 총 {len(self.data_pairs)} 슬라이스")
        print(f"   라벨 분포: {dict(label_counts)}")

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        image, label = self.data_pairs[idx]

        # NumPy to Tensor
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)  # (1, H, W)
        label = torch.tensor(label, dtype=torch.long)  # (H, W)

        # Resize
        image = image.unsqueeze(0)  # (1, 1, H, W)
        label = label.unsqueeze(0).unsqueeze(0).float()  # (1, 1, H, W)

        image = F.interpolate(image, size=self.target_size, mode='bilinear', align_corners=False)
        label = F.interpolate(label, size=self.target_size, mode='nearest')

        image = image.squeeze(0)  # (1, H, W)
        label = label.squeeze(0).squeeze(0).long()  # (H, W)

        return image, label

def collate_fn(batch):
    images = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    return torch.stack(images), torch.stack(labels)

In [ ]:
# 5. 데이터셋 생성
print("\n--- 순수 2D Dataset 생성 중 ---")
dataset = Pure2DSliceDataset(img_tr_paths, lbl_tr_paths, target_size=(256, 256))

In [ ]:
# Train/Validation split (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f"\n📊 데이터 분할:")
print(f"   Train: {train_size} 슬라이스")
print(f"   Validation: {val_size} 슬라이스")

In [ ]:
# 배치 확인
for images, labels in train_loader:
    print(f"\n✓ 배치 shape 확인:")
    print(f"   Images: {images.shape}")
    print(f"   Labels: {labels.shape}")
    print(f"   고유 라벨: {torch.unique(labels).tolist()}")
    break

In [ ]:
# 6. 2D U-Net 모델 정의
class UNet2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=3):
        super(UNet2D, self).__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.enc1 = conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = conv_block(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = conv_block(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = conv_block(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)

        # Output
        self.out = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        # Bottleneck
        b = self.bottleneck(self.pool4(e4))

        # Decoder
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out(d1)

In [ ]:
# 7. 평가 메트릭 함수
def calculate_metrics(pred, target, num_classes=3):
    """Dice Score, IoU, Confusion Matrix 계산"""
    pred_classes = torch.argmax(pred, dim=1).cpu().numpy().flatten()
    target = target.cpu().numpy().flatten()

    # Confusion Matrix
    cm = confusion_matrix(target, pred_classes, labels=list(range(num_classes)))

    # Per-class metrics
    dice_scores = []
    iou_scores = []

    for c in range(num_classes):
        pred_c = (pred_classes == c)
        target_c = (target == c)

        intersection = np.sum(pred_c & target_c)
        union = np.sum(pred_c | target_c)
        pred_sum = np.sum(pred_c)
        target_sum = np.sum(target_c)

        # Dice Score
        dice = (2. * intersection + 1e-6) / (pred_sum + target_sum + 1e-6)
        dice_scores.append(dice)

        # IoU (Jaccard)
        iou = (intersection + 1e-6) / (union + 1e-6)
        iou_scores.append(iou)

    return {
        'dice': dice_scores,
        'iou': iou_scores,
        'confusion_matrix': cm
    }

def plot_confusion_matrix(cm, class_names, epoch, save_path=None):
    """Confusion Matrix 시각화 (히트맵)"""
    plt.figure(figsize=(10, 8))

    # 정규화된 confusion matrix 계산 (percentage)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    # 히트맵 그리기
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Percentage (%)'}, vmin=0, vmax=100)

    # 각 셀에 실제 개수도 표시
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j + 0.5, i + 0.7, f'({int(cm[i, j])})',
                    ha='center', va='center', fontsize=9, color='gray')

    plt.title(f'Confusion Matrix - Epoch {epoch}\n(Percentage with actual counts)',
              fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 Confusion Matrix 저장: {save_path}")
    plt.show()
    plt.close()

def plot_dice_scores(train_dice_history, val_dice_history, class_names, save_path=None):
    """클래스별 Dice Score 시각화"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    colors = ['#2ecc71', '#e74c3c', '#3498db']

    # 각 클래스별 Dice Score
    for i, (ax, class_name, color) in enumerate(zip(axes.flat[:3], class_names, colors)):
        train_scores = [d[i] for d in train_dice_history]
        val_scores = [d[i] for d in val_dice_history]

        ax.plot(train_scores, label='Train', marker='o', linewidth=2, color=color, alpha=0.7)
        ax.plot(val_scores, label='Validation', marker='s', linewidth=2,
                color=color, linestyle='--', alpha=0.9)
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Dice Score', fontsize=11)
        ax.set_title(f'{class_name} Dice Score', fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])

    # 평균 Dice Score (Hippocampi only)
    ax = axes.flat[3]
    train_mean = [np.mean(d[1:]) for d in train_dice_history]
    val_mean = [np.mean(d[1:]) for d in val_dice_history]

    ax.plot(train_mean, label='Train', marker='o', linewidth=2, color='#9b59b6', alpha=0.7)
    ax.plot(val_mean, label='Validation', marker='s', linewidth=2,
            color='#9b59b6', linestyle='--', alpha=0.9)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Mean Dice Score', fontsize=11)
    ax.set_title('Mean Hippocampus Dice Score', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 Dice Score 그래프 저장: {save_path}")
    plt.show()
    plt.close()

In [ ]:
# 8. 모델 초기화
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n🖥️ Device: {device}")

model = UNet2D(in_channels=1, out_channels=3).to(device)
print(f"✅ 2D U-Net 생성 완료")
print(f"   파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 클래스 가중치 계산
all_labels_for_weight = []
for _, lbl in dataset:
    all_labels_for_weight.extend(lbl.numpy().flatten())

label_counts = Counter(all_labels_for_weight)
total_pixels = sum(label_counts.values())
class_weights = []
for i in range(3):
    if i in label_counts:
        weight = total_pixels / (3 * label_counts[i])
    else:
        weight = 1.0
    class_weights.append(weight)

class_weights = torch.FloatTensor(class_weights).to(device)
print(f"\n⚖️ 클래스 가중치: {[f'{w:.4f}' for w in class_weights.tolist()]}")

In [ ]:
# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

In [ ]:
# 9. 학습 루프
num_epochs = 30
best_val_dice = 0.0
class_names = ['Background', 'Left Hippo', 'Right Hippo']

print("\n" + "="*70)
print("🚀 학습 시작 (순수 2D U-Net)")
print("="*70)

train_losses = []
val_losses = []
train_dice_history = []
val_dice_history = []

In [ ]:
for epoch in range(num_epochs):
        # ============ Training ============
    model.train()
    train_loss = 0
    train_metrics_list = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        metrics = calculate_metrics(outputs, labels)
        train_metrics_list.append(metrics)

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # 평균 메트릭 계산
    train_dice = np.mean([m['dice'] for m in train_metrics_list], axis=0)
    train_iou = np.mean([m['iou'] for m in train_metrics_list], axis=0)
    train_dice_history.append(train_dice)

In [ ]:
    # ============ Validation ============
    model.eval()
    val_loss = 0
    val_metrics_list = []
    all_cm = np.zeros((3, 3))

    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            metrics = calculate_metrics(outputs, labels)
            val_metrics_list.append(metrics)
            all_cm += metrics['confusion_matrix']

            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    val_dice = np.mean([m['dice'] for m in val_metrics_list], axis=0)
    val_iou = np.mean([m['iou'] for m in val_metrics_list], axis=0)
    val_dice_history.append(val_dice)

    # Learning rate scheduling
    scheduler.step(avg_val_loss)

In [ ]:
    # ============ Epoch Summary ============
    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{num_epochs} Summary:")
    print(f"{'='*70}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"\nTrain Metrics:")
    for i, name in enumerate(class_names):
        print(f"  {name:15s} - Dice: {train_dice[i]:.4f} | IoU: {train_iou[i]:.4f}")
    print(f"  {'Mean (Hippo)':15s} - Dice: {np.mean(train_dice[1:]):.4f}")

    print(f"\nValidation Metrics:")
    for i, name in enumerate(class_names):
        print(f"  {name:15s} - Dice: {val_dice[i]:.4f} | IoU: {val_iou[i]:.4f}")
    print(f"  {'Mean (Hippo)':15s} - Dice: {np.mean(val_dice[1:]):.4f}")

In [ ]:
    # Confusion Matrix 출력 (5 에포크마다)
    if (epoch + 1) % 5 == 0:
        print(f"\n{'='*70}")
        print(f"📊 Confusion Matrix 시각화 (Epoch {epoch+1})")
        print(f"{'='*70}")

        # Confusion Matrix 이미지 생성 및 저장
        cm_save_path = f'/content/confusion_matrix_epoch_{epoch+1}.png'
        plot_confusion_matrix(all_cm, class_names, epoch+1, save_path=cm_save_path)

        # 콘솔에도 텍스트로 출력
        print(f"\nConfusion Matrix (Raw Counts):")
        print(f"{'':15s}", end='')
        for name in class_names:
            print(f"{name:>15s}", end='')
        print()
        for i, name in enumerate(class_names):
            print(f"{name:15s}", end='')
            for j in range(3):
                print(f"{int(all_cm[i, j]):>15d}", end='')
            print()

In [ ]:
    # Dice Score 그래프 (10 에포크마다)
    if (epoch + 1) % 10 == 0:
        print(f"\n{'='*70}")
        print(f"📈 Dice Score 시각화 (Epoch {epoch+1})")
        print(f"{'='*70}")

        dice_save_path = f'/content/dice_scores_epoch_{epoch+1}.png'
        plot_dice_scores(train_dice_history, val_dice_history, class_names,
                        save_path=dice_save_path)

In [ ]:
    # Best model 저장
    val_hippo_dice = np.mean(val_dice[1:])
    if val_hippo_dice > best_val_dice:
        best_val_dice = val_hippo_dice
        torch.save(model.state_dict(), '/content/best_unet2d_model.pth')
        print(f"\n🎯 Best model saved! (Val Hippo Dice: {best_val_dice:.4f})")

print("\n" + "="*70)
print("✅ 학습 완료!")
print(f"🏆 Best Validation Dice (Hippocampi): {best_val_dice:.4f}")
print("="*70)

In [ ]:
# 10. 최종 Confusion Matrix & Dice Score 시각화
print("\n" + "="*70)
print("📊 최종 결과 시각화")
print("="*70)

# 최종 Confusion Matrix
model.load_state_dict(torch.load('/content/best_unet2d_model.pth'))
model.eval()

final_cm = np.zeros((3, 3))
final_metrics_list = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="최종 평가"):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        metrics = calculate_metrics(outputs, labels)
        final_metrics_list.append(metrics)
        final_cm += metrics['confusion_matrix']

final_dice = np.mean([m['dice'] for m in final_metrics_list], axis=0)
final_iou = np.mean([m['iou'] for m in final_metrics_list], axis=0)

print("\n🎯 최종 검증 세트 성능:")
for i, name in enumerate(class_names):
    print(f"  {name:15s} - Dice: {final_dice[i]:.4f} | IoU: {final_iou[i]:.4f}")
print(f"  {'Mean (Hippo)':15s} - Dice: {np.mean(final_dice[1:]):.4f}")

In [ ]:
# 최종 Confusion Matrix 이미지
print("\n🖼️ 최종 Confusion Matrix 생성 중...")
plot_confusion_matrix(final_cm, class_names, epoch="Final",
                     save_path='/content/final_confusion_matrix.png')

In [ ]:
# 최종 Dice Score 그래프
print("\n📈 최종 Dice Score 그래프 생성 중...")
plot_dice_scores(train_dice_history, val_dice_history, class_names,
                save_path='/content/final_dice_scores.png')

In [ ]:
# 11. 학습 곡선 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
axes[0].plot(train_losses, label='Train Loss', marker='o')
axes[0].plot(val_losses, label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Dice curve (Hippocampi only)
train_hippo_dice = [np.mean(d[1:]) for d in train_dice_history]
val_hippo_dice = [np.mean(d[1:]) for d in val_dice_history]
axes[1].plot(train_hippo_dice, label='Train Dice (Hippo)', marker='o')
axes[1].plot(val_hippo_dice, label='Val Dice (Hippo)', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Score')
axes[1].set_title('Hippocampus Dice Score')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print("\n" + "="*70)
print("💾 저장된 파일들:")
print("="*70)
print("  📈 /content/training_curves.png - 학습/검증 손실 & Dice 곡선")
print("  🖼️ /content/final_confusion_matrix.png - 최종 Confusion Matrix")
print("  📊 /content/final_dice_scores.png - 클래스별 Dice Score 추이")
print("  💾 /content/best_unet2d_model.pth - 최고 성능 모델 가중치")
print("  🔍 /content/confusion_matrix_epoch_*.png - 5 에포크마다 저장된 CM")
print("  📉 /content/dice_scores_epoch_*.png - 10 에포크마다 저장된 Dice 그래프")
print("="*70)

In [ ]:
# 12. 예측 결과 시각화
print("\n" + "="*70)
print("🔮 예측 결과 시각화")
print("="*70)

def visualize_predictions(model, dataset, device, num_samples=12, save_path=None):
    """예측 결과 시각화 (원본 이미지, GT, 예측)"""
    model.eval()

    # 랜덤 샘플 선택
    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    # 4x3 그리드 (각 행: 이미지, GT, 예측)
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for idx, sample_idx in enumerate(indices):
            image, label = dataset[sample_idx]

            # 배치 차원 추가
            image_input = image.unsqueeze(0).to(device)  # (1, 1, H, W)

            # 예측
            output = model(image_input)
            pred = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()  # (H, W)

            # 시각화용 데이터 준비
            img_np = image.squeeze(0).cpu().numpy()  # (H, W)
            label_np = label.cpu().numpy()  # (H, W)

            # 1) 원본 이미지
            axes[idx, 0].imshow(img_np, cmap='gray')
            axes[idx, 0].set_title(f'Sample {sample_idx} - Input Image', fontweight='bold')
            axes[idx, 0].axis('off')

            # 2) Ground Truth
            axes[idx, 1].imshow(img_np, cmap='gray')
            # 해마 영역만 컬러로 오버레이
            gt_overlay = np.zeros((*label_np.shape, 4))
            gt_overlay[label_np == 1] = [1, 0, 0, 0.5]  # Left Hippo - Red
            gt_overlay[label_np == 2] = [0, 0, 1, 0.5]  # Right Hippo - Blue
            axes[idx, 1].imshow(gt_overlay)
            axes[idx, 1].set_title('Ground Truth\n(Red: Left, Blue: Right)', fontweight='bold')
            axes[idx, 1].axis('off')

            # 3) 예측 결과
            axes[idx, 2].imshow(img_np, cmap='gray')
            # 예측된 해마 영역 오버레이
            pred_overlay = np.zeros((*pred.shape, 4))
            pred_overlay[pred == 1] = [1, 0, 0, 0.5]  # Left Hippo - Red
            pred_overlay[pred == 2] = [0, 0, 1, 0.5]  # Right Hippo - Blue
            axes[idx, 2].imshow(pred_overlay)

            # Dice Score 계산
            dice_left = 2 * np.sum((pred == 1) & (label_np == 1)) / (np.sum(pred == 1) + np.sum(label_np == 1) + 1e-6)
            dice_right = 2 * np.sum((pred == 2) & (label_np == 2)) / (np.sum(pred == 2) + np.sum(label_np == 2) + 1e-6)

            axes[idx, 2].set_title(f'Prediction\nDice - L: {dice_left:.3f}, R: {dice_right:.3f}',
                                  fontweight='bold')
            axes[idx, 2].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 예측 결과 저장: {save_path}")
    plt.show()
    plt.close()

In [ ]:
# Validation 세트에서 샘플링하여 시각화
print("\n🖼️ Validation 세트 예측 결과 생성 중...")
visualize_predictions(model, val_dataset.dataset, device, num_samples=8,
                     save_path='/content/prediction_samples.png')

In [ ]:
# 13. 3D 볼륨 전체 예측 시각화 (한 케이스의 모든 슬라이스)
print("\n" + "="*70)
print("🎬 3D 볼륨 예측 시각화 (연속 슬라이스)")
print("="*70)

def visualize_3d_volume_prediction(model, img_path, lbl_path, device, target_size=(256, 256), save_path=None):
    """3D 볼륨의 모든 슬라이스에 대한 예측 결과 시각화"""
    model.eval()

    # 3D 볼륨 로드
    img_nii = nib.load(img_path)
    lbl_nii = nib.load(lbl_path)
    img_volume = img_nii.get_fdata()
    lbl_volume = lbl_nii.get_fdata()

    # 해마가 있는 슬라이스만 추출
    hippo_slices = []
    for z in range(img_volume.shape[2]):
        if np.sum(lbl_volume[:, :, z] > 0) > 0:
            hippo_slices.append(z)

    num_slices = len(hippo_slices)
    if num_slices == 0:
        print("해마가 없는 볼륨입니다.")
        return

    # 그리드 크기 계산 (최대 4x4)
    cols = 4
    rows = min(4, (num_slices + cols - 1) // cols)
    selected_slices = hippo_slices[::max(1, num_slices // (rows * cols))][:rows * cols]

    fig, axes = plt.subplots(rows, cols * 3, figsize=(5 * cols, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)

    case_name = os.path.basename(img_path).replace('.nii.gz', '')
    fig.suptitle(f'3D Volume Prediction - {case_name}', fontsize=16, fontweight='bold')

    with torch.no_grad():
        for idx, z in enumerate(selected_slices):
            row = idx // cols
            col_start = (idx % cols) * 3

            # 슬라이스 추출 및 전처리
            img_slice = img_volume[:, :, z]
            lbl_slice = lbl_volume[:, :, z]

            if img_slice.max() > 0:
                img_slice = (img_slice - img_slice.mean()) / (img_slice.std() + 1e-8)

            # Tensor 변환 및 Resize
            img_tensor = torch.tensor(img_slice, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            img_tensor = F.interpolate(img_tensor, size=target_size, mode='bilinear', align_corners=False)
            img_tensor = img_tensor.to(device)

            # 예측
            output = model(img_tensor)
            pred = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

            img_display = img_tensor.squeeze().cpu().numpy()

            # 1) 원본 이미지
            axes[row, col_start].imshow(img_display, cmap='gray')
            axes[row, col_start].set_title(f'Slice {z}', fontsize=10)
            axes[row, col_start].axis('off')

            # 2) Ground Truth
            axes[row, col_start + 1].imshow(img_display, cmap='gray')
            lbl_resized = F.interpolate(
                torch.tensor(lbl_slice).unsqueeze(0).unsqueeze(0).float(),
                size=target_size, mode='nearest'
            ).squeeze().numpy()

            gt_overlay = np.zeros((*lbl_resized.shape, 4))
            gt_overlay[lbl_resized == 1] = [1, 0, 0, 0.5]
            gt_overlay[lbl_resized == 2] = [0, 0, 1, 0.5]
            axes[row, col_start + 1].imshow(gt_overlay)
            axes[row, col_start + 1].set_title('GT', fontsize=10)
            axes[row, col_start + 1].axis('off')

            # 3) 예측
            axes[row, col_start + 2].imshow(img_display, cmap='gray')
            pred_overlay = np.zeros((*pred.shape, 4))
            pred_overlay[pred == 1] = [1, 0, 0, 0.5]
            pred_overlay[pred == 2] = [0, 0, 1, 0.5]
            axes[row, col_start + 2].imshow(pred_overlay)
            axes[row, col_start + 2].set_title('Pred', fontsize=10)
            axes[row, col_start + 2].axis('off')

    # 남은 서브플롯 숨기기
    for idx in range(len(selected_slices), rows * cols):
        row = idx // cols
        for col in range(3):
            axes[row, (idx % cols) * 3 + col].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   💾 3D 볼륨 예측 저장: {save_path}")
    plt.show()
    plt.close()

In [ ]:
# 첫 번째 케이스로 3D 볼륨 시각화
print("\n🎥 첫 번째 케이스의 3D 볼륨 예측 중...")
if len(img_tr_paths) > 0:
    visualize_3d_volume_prediction(
        model,
        img_tr_paths[0],
        lbl_tr_paths[0],
        device,
        save_path='/content/3d_volume_prediction_case1.png'
    )

# 14. 최종 요약
print("\n" + "="*70)
print("✅ 모든 분석 완료!")
print("="*70)
print("\n📁 생성된 모든 파일:")
print("  " + "-"*66)
print("  📊 메트릭 & 평가:")
print("     • final_confusion_matrix.png - 최종 혼동 행렬")
print("     • final_dice_scores.png - 클래스별 Dice Score 추이")
print("     • training_curves.png - Loss & Dice 학습 곡선")
print("  " + "-"*66)
print("  🔮 예측 결과:")
print("     • prediction_samples.png - Validation 샘플 예측")
print("     • 3d_volume_prediction_case1.png - 첫 케이스 전체 슬라이스")
print("  " + "-"*66)
print("  💾 모델 & 중간 결과:")
print("     • best_unet2d_model.pth - 최고 성능 모델")
print("     • confusion_matrix_epoch_*.png - 에포크별 CM")
print("     • dice_scores_epoch_*.png - 에포크별 Dice")
print("  " + "-"*66)
print(f"\n🏆 최종 성능: {best_val_dice:.4f} (Hippocampus Mean Dice)")
print("="*70)